# Options Pricing Model: Black-Scholes & Monte Carlo Simulation

**Author:** Harish Bodasinghi  
**Tools:** Python, NumPy, SciPy, Matplotlib, Seaborn  
**Topics:** Quantitative Finance, Stochastic Modelling, Risk Analytics

---

## Overview

This project implements two approaches to pricing European options:

1. **Black-Scholes Analytical Model** — closed-form solution for European call/put pricing
2. **Monte Carlo Simulation** — stochastic simulation of stock price paths using Geometric Brownian Motion (GBM)
3. **Greeks Analysis** — Delta, Gamma, Theta, Vega sensitivity measures
4. **Profit & Loss (PnL) Surface** — visualizing option value across strike price and volatility

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import norm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['text.color'] = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['grid.color'] = '#21262d'
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['font.family'] = 'monospace'

np.random.seed(42)
print('Libraries loaded successfully.')

## 2. Black-Scholes Model

The Black-Scholes formula prices European options assuming:
- Stock follows **Geometric Brownian Motion**
- No dividends, constant volatility, continuous trading

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

Where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
class BlackScholes:
    """
    Black-Scholes Option Pricing Model.
    Prices European Call and Put options and computes the Greeks.
    """

    def __init__(self, S, K, T, r, sigma):
        """
        Parameters:
        -----------
        S     : float  - Current stock price
        K     : float  - Strike price
        T     : float  - Time to expiry in years
        r     : float  - Risk-free interest rate (annual)
        sigma : float  - Volatility (annual)
        """
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        self.d2 = self.d1 - sigma * np.sqrt(T)

    def call_price(self):
        return self.S * norm.cdf(self.d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)

    def put_price(self):
        return self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2) - self.S * norm.cdf(-self.d1)

    # --- Greeks ---
    def delta(self, option='call'):
        """Rate of change of option price w.r.t. stock price."""
        if option == 'call':
            return norm.cdf(self.d1)
        return norm.cdf(self.d1) - 1

    def gamma(self):
        """Rate of change of delta w.r.t. stock price."""
        return norm.pdf(self.d1) / (self.S * self.sigma * np.sqrt(self.T))

    def theta(self, option='call'):
        """Rate of change of option price w.r.t. time (per day)."""
        term1 = -(self.S * norm.pdf(self.d1) * self.sigma) / (2 * np.sqrt(self.T))
        if option == 'call':
            return (term1 - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)) / 365
        return (term1 + self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2)) / 365

    def vega(self):
        """Sensitivity to volatility (per 1% move in vol)."""
        return self.S * norm.pdf(self.d1) * np.sqrt(self.T) * 0.01

    def summary(self):
        print('=' * 50)
        print('   BLACK-SCHOLES OPTION PRICING SUMMARY')
        print('=' * 50)
        print(f'  Stock Price (S)     : ${self.S:.2f}')
        print(f'  Strike Price (K)    : ${self.K:.2f}')
        print(f'  Time to Expiry (T)  : {self.T:.2f} years')
        print(f'  Risk-Free Rate (r)  : {self.r*100:.1f}%')
        print(f'  Volatility (sigma)  : {self.sigma*100:.1f}%')
        print('-' * 50)
        print(f'  CALL Price          : ${self.call_price():.4f}')
        print(f'  PUT  Price          : ${self.put_price():.4f}')
        print('-' * 50)
        print('  GREEKS (CALL):')
        print(f'    Delta             : {self.delta("call"):.4f}')
        print(f'    Gamma             : {self.gamma():.4f}')
        print(f'    Theta (per day)   : ${self.theta("call"):.4f}')
        print(f'    Vega  (per 1% vol): ${self.vega():.4f}')
        print('=' * 50)


# --- Run with example parameters ---
bs = BlackScholes(S=100, K=105, T=1.0, r=0.05, sigma=0.20)
bs.summary()

## 3. Monte Carlo Simulation — Stock Price Paths

Using **Geometric Brownian Motion (GBM)**:

$$S_t = S_0 \cdot \exp\left[(r - \frac{\sigma^2}{2})t + \sigma W_t\right]$$

Where $W_t$ is a Wiener process (standard Brownian motion).

In [ ]:
def monte_carlo_option_price(S, K, T, r, sigma, n_simulations=100_000, n_steps=252, option='call'):
    """
    Prices a European option using Monte Carlo simulation.
    Models stock price paths via Geometric Brownian Motion.
    """
    dt = T / n_steps
    # Simulate all paths at once using vectorized operations
    Z = np.random.standard_normal((n_simulations, n_steps))
    drift = (r - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    log_returns = drift + diffusion
    price_paths = S * np.exp(np.cumsum(log_returns, axis=1))

    # Final prices
    S_T = price_paths[:, -1]

    # Payoffs
    if option == 'call':
        payoffs = np.maximum(S_T - K, 0)
    else:
        payoffs = np.maximum(K - S_T, 0)

    # Discounted expected payoff
    price = np.exp(-r * T) * np.mean(payoffs)
    std_error = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_simulations)

    return price, std_error, price_paths


# --- Parameters ---
S, K, T, r, sigma = 100, 105, 1.0, 0.05, 0.20
n_sim = 100_000

mc_call, se_call, paths = monte_carlo_option_price(S, K, T, r, sigma, n_sim, option='call')
mc_put, se_put, _       = monte_carlo_option_price(S, K, T, r, sigma, n_sim, option='put')
bs_model = BlackScholes(S, K, T, r, sigma)

print('=' * 55)
print('   MONTE CARLO vs BLACK-SCHOLES COMPARISON')
print('=' * 55)
print(f'  Simulations         : {n_sim:,}')
print(f'  Steps per path      : 252 (trading days)')
print('-' * 55)
print(f'  CALL  — Black-Scholes : ${bs_model.call_price():.4f}')
print(f'  CALL  — Monte Carlo   : ${mc_call:.4f}  (±{se_call:.4f})')
print(f'  CALL  — Difference    : ${abs(mc_call - bs_model.call_price()):.4f}')
print('-' * 55)
print(f'  PUT   — Black-Scholes : ${bs_model.put_price():.4f}')
print(f'  PUT   — Monte Carlo   : ${mc_put:.4f}  (±{se_put:.4f})')
print(f'  PUT   — Difference    : ${abs(mc_put - bs_model.put_price()):.4f}')
print('=' * 55)

## 4. Visualizations

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
TEAL = '#1f6feb'
ORANGE = '#f78166'
GREEN = '#3fb950'
PURPLE = '#bc8cff'

# ── Plot 1: Simulated Stock Price Paths ──
ax1 = fig.add_subplot(gs[0, 0])
t_axis = np.linspace(0, T, 252)
for i in range(200):
    ax1.plot(t_axis, paths[i], alpha=0.08, linewidth=0.5, color=TEAL)
# Highlight a few paths
for i in range(5):
    ax1.plot(t_axis, paths[i*20], alpha=0.9, linewidth=1.2, color=ORANGE)
ax1.axhline(y=K, color=GREEN, linestyle='--', linewidth=1.5, label=f'Strike K=${K}')
ax1.axhline(y=S, color='#8b949e', linestyle=':', linewidth=1, label=f'Spot S=${S}')
ax1.set_title('Simulated GBM Stock Price Paths', color='#e6edf3', fontsize=11, pad=10)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Stock Price ($)')
ax1.legend(fontsize=8)
ax1.grid(True)

# ── Plot 2: Distribution of Terminal Prices ──
ax2 = fig.add_subplot(gs[0, 1])
S_T = paths[:, -1]
ax2.hist(S_T, bins=80, color=TEAL, alpha=0.7, edgecolor='none', density=True)
ax2.axvline(x=K, color=GREEN, linestyle='--', linewidth=2, label=f'Strike K=${K}')
ax2.axvline(x=np.mean(S_T), color=ORANGE, linestyle='-', linewidth=2, label=f'Mean=${np.mean(S_T):.1f}')
# Overlay lognormal fit
x_range = np.linspace(S_T.min(), S_T.max(), 300)
mu_fit = np.mean(np.log(S_T))
sigma_fit = np.std(np.log(S_T))
pdf_fit = stats.lognorm.pdf(x_range, s=sigma_fit, scale=np.exp(mu_fit))
ax2.plot(x_range, pdf_fit, color=PURPLE, linewidth=2, label='Log-normal fit')
ax2.set_title('Terminal Stock Price Distribution', color='#e6edf3', fontsize=11, pad=10)
ax2.set_xlabel('Stock Price at Expiry ($)')
ax2.set_ylabel('Density')
ax2.legend(fontsize=8)
ax2.grid(True)

# ── Plot 3: Option Price vs Stock Price (Delta visualization) ──
ax3 = fig.add_subplot(gs[1, 0])
spot_range = np.linspace(60, 160, 200)
call_prices = [BlackScholes(s, K, T, r, sigma).call_price() for s in spot_range]
put_prices  = [BlackScholes(s, K, T, r, sigma).put_price()  for s in spot_range]
intrinsic_call = np.maximum(spot_range - K, 0)
ax3.plot(spot_range, call_prices, color=GREEN,  linewidth=2.5, label='Call Price (BS)')
ax3.plot(spot_range, put_prices,  color=ORANGE, linewidth=2.5, label='Put Price (BS)')
ax3.plot(spot_range, intrinsic_call, color='#8b949e', linewidth=1, linestyle='--', label='Intrinsic Value (Call)')
ax3.axvline(x=S, color=TEAL, linestyle=':', linewidth=1.5, label=f'Current S=${S}')
ax3.axvline(x=K, color=PURPLE, linestyle=':', linewidth=1.5, label=f'Strike K=${K}')
ax3.set_title('Option Price vs Stock Price', color='#e6edf3', fontsize=11, pad=10)
ax3.set_xlabel('Stock Price ($)')
ax3.set_ylabel('Option Price ($)')
ax3.legend(fontsize=8)
ax3.grid(True)

# ── Plot 4: Volatility Smile — IV across strikes ──
ax4 = fig.add_subplot(gs[1, 1])
strikes = np.linspace(75, 135, 50)
# Simulate a volatility smile shape
atm_vol = 0.20
smile_vol = atm_vol + 0.00015 * (strikes - S)**2 / S  # quadratic smile
skew_vol  = atm_vol - 0.0008 * (strikes - S)           # linear skew
ax4.plot(strikes, smile_vol * 100, color=GREEN,  linewidth=2.5, label='Volatility Smile')
ax4.plot(strikes, skew_vol  * 100, color=ORANGE, linewidth=2.5, linestyle='--', label='Volatility Skew')
ax4.axhline(y=atm_vol*100, color='#8b949e', linewidth=1, linestyle=':', label=f'ATM Vol {atm_vol*100:.0f}%')
ax4.axvline(x=S, color=TEAL, linewidth=1.5, linestyle=':', label=f'ATM S=${S}')
ax4.set_title('Implied Volatility Surface (Smile/Skew)', color='#e6edf3', fontsize=11, pad=10)
ax4.set_xlabel('Strike Price ($)')
ax4.set_ylabel('Implied Volatility (%)')
ax4.legend(fontsize=8)
ax4.grid(True)

fig.suptitle('Options Pricing: Black-Scholes + Monte Carlo Analysis', 
             color='#e6edf3', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('options_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Chart saved as options_analysis.png')

## 5. Greeks Sensitivity Analysis

In [ ]:
# Greeks across a range of stock prices
spot_range = np.linspace(60, 160, 300)
models = [BlackScholes(s, K, T, r, sigma) for s in spot_range]

deltas_call = [m.delta('call') for m in models]
deltas_put  = [m.delta('put')  for m in models]
gammas      = [m.gamma()       for m in models]
vegas       = [m.vega()        for m in models]
thetas_call = [m.theta('call') for m in models]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('#0d1117')

plots = [
    (axes[0,0], deltas_call, deltas_put, 'Delta', 'Call Delta', 'Put Delta'),
    (axes[0,1], gammas,      None,       'Gamma', 'Gamma',      None),
    (axes[1,0], vegas,       None,       'Vega (per 1% vol)', 'Vega', None),
    (axes[1,1], thetas_call, None,       'Theta (per day, $)', 'Call Theta', None),
]

for ax, y1, y2, title, l1, l2 in plots:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
    ax.grid(color='#21262d', alpha=0.5)
    ax.plot(spot_range, y1, color=GREEN, linewidth=2, label=l1)
    if y2:
        ax.plot(spot_range, y2, color=ORANGE, linewidth=2, label=l2)
    ax.axvline(x=K, color='#8b949e', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(title, color='#e6edf3', fontsize=11)
    ax.set_xlabel('Stock Price ($)', color='#8b949e')
    ax.legend(fontsize=9)

fig.suptitle('Option Greeks Sensitivity Analysis', color='#e6edf3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('greeks_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Greeks chart saved.')

## 6. Convergence Analysis — Monte Carlo Accuracy vs Simulations

In [ ]:
# Show how MC price converges to BS price as simulations increase
sim_counts = [100, 500, 1000, 5000, 10000, 50000, 100000]
mc_prices, mc_errors = [], []

for n in sim_counts:
    price, se, _ = monte_carlo_option_price(S, K, T, r, sigma, n_simulations=n, option='call')
    mc_prices.append(price)
    mc_errors.append(se)

bs_call = bs_model.call_price()

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
ax.tick_params(colors='#8b949e')
ax.grid(color='#21262d', alpha=0.5)

ax.semilogx(sim_counts, mc_prices, 'o-', color=TEAL, linewidth=2, markersize=7, label='MC Price')
ax.fill_between(sim_counts,
                [p - e for p, e in zip(mc_prices, mc_errors)],
                [p + e for p, e in zip(mc_prices, mc_errors)],
                alpha=0.2, color=TEAL, label='±1 Std Error')
ax.axhline(y=bs_call, color=GREEN, linestyle='--', linewidth=2, label=f'Black-Scholes: ${bs_call:.4f}')
ax.set_title('Monte Carlo Convergence to Black-Scholes Price', color='#e6edf3', fontsize=12)
ax.set_xlabel('Number of Simulations (log scale)', color='#8b949e')
ax.set_ylabel('Call Option Price ($)', color='#8b949e')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('mc_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f'\nBlack-Scholes Call Price : ${bs_call:.6f}')
print(f'Monte Carlo  Call Price  : ${mc_prices[-1]:.6f}  (100k sims)')
print(f'Difference               : ${abs(mc_prices[-1]-bs_call):.6f}')

## 7. Key Takeaways

| Metric | Black-Scholes | Monte Carlo (100k) |
|--------|--------------|--------------------|
| Call Price | Closed-form exact | Converges with more sims |
| Put Price  | Closed-form exact | Converges with more sims |
| Computation | Instant | Scales with simulations |
| Flexibility | European options only | Any payoff structure |
| Greeks | Analytical | Requires finite-differencing |

### Insights:
- **Black-Scholes** is fast and exact for European options under its assumptions
- **Monte Carlo** is flexible and can price exotic options (Asian, Barrier) — at computational cost
- **Greeks** (Delta, Gamma, Vega, Theta) are essential for hedging and risk management
- **Volatility Smile/Skew** shows market-implied vol differs from constant BS assumption — real markets price in tail risk
- MC price converges to BS price as simulations → ∞ (Law of Large Numbers)

---

## Extensions (Future Work)
- Implied Volatility surface from real market data (Yahoo Finance / CBOE)
- American option pricing using Least Squares Monte Carlo (Longstaff-Schwartz)
- Heston Stochastic Volatility Model
- VaR and CVaR computation for options portfolios

---
*Built by Harish Bodasinghi | Data Scientist | Quantitative Finance*